In [21]:
import os
import shutil
from pathlib import Path
import pandas as pd

In [22]:
PATH = "../data/2023/"

In [23]:
os.makedirs(os.path.join(PATH, "step4_image_feature_extract"), exist_ok=True)

In [24]:
# Nén thư mục step0_clean_data/filtered_images sang step4_image_feature/filtered_images.tar.gz
shutil.make_archive(
    os.path.join(PATH, "step4_image_feature_extract", "filtered_images"),
    "gztar",
    os.path.join(PATH, "step0_clean_data", "filtered_images"),
)

'c:\\projects\\MMRec\\preprocessing\\data\\2023\\step4_image_feature_extract\\filtered_images.tar.gz'

In [25]:
# Giải nén thư mục filtered_images.tar.gz vào step4_image_feature_extract/filtered_images
shutil.unpack_archive(
    os.path.join(PATH, "step4_image_feature_extract", "filtered_images.tar.gz"),
    os.path.join(PATH, "step4_image_feature_extract", "filtered_images"),
)

## Kiểm tra lại thư mục filtered_images xem có thiếu hình nào sau khi đã thử tải lại không

In [26]:
df_retry_image = pd.read_parquet(os.path.join(PATH, "step0_clean_data", "df_miss_image.parquet"))

In [27]:
df_meta_with_image = pd.read_parquet(os.path.join(PATH, "step0_clean_data", "df_meta.with_image.parquet"))

In [28]:
# Đọc lại thư mục filtered_images và kiểm tra số hình đã copy
filtered_images_dir = Path(PATH) / "step0_clean_data" / "filtered_images"
if filtered_images_dir.exists():
    image_paths = [p for p in filtered_images_dir.rglob("*") if p.is_file()]
    total_images = len(image_paths)
    print(f"✅ Tổng file hình trong filtered_images: {total_images}")

    if "asin" in df_meta_with_image.columns:
        copied_asins = {p.stem for p in image_paths}
        asin_in_df = df_meta_with_image["asin"].dropna().astype(str).str.strip()
        total_asins_in_df = asin_in_df.nunique()
        copied_asins_count = len(copied_asins)
        missing_asins = sorted(set(asin_in_df) - copied_asins)

        print(f"ℹ️ Unique ASIN trong df: {total_asins_in_df}")
        print(f"ℹ️ Unique ASIN đã copy vào filtered_images: {copied_asins_count}")
        print(f"ℹ️ ASIN trong df chưa có file trong filtered_images: {len(missing_asins)}")
        if missing_asins:
            print("Các ASIN thiếu (max 50):")
            print(missing_asins[:50])
else:
    print(f"⚠️ Thư mục filtered_images không tồn tại: {filtered_images_dir}")

✅ Tổng file hình trong filtered_images: 35979
ℹ️ Unique ASIN trong df: 35997
ℹ️ Unique ASIN đã copy vào filtered_images: 35979
ℹ️ ASIN trong df chưa có file trong filtered_images: 18
Các ASIN thiếu (max 50):
['B00005QI1H', 'B0017WEH1S', 'B0018CJ7G2', 'B00192H1KA', 'B004AAIQ1G', 'B0092KKTQ4', 'B009WUPGQC', 'B00BX8RRJA', 'B00OPZR8L0', 'B079659G3W', 'B07TLHM5S8', 'B085QPYTJ8', 'B08XYQL2VF', 'B099J4VRS2', 'B09JP1HXN8', 'B09JP21WCK', 'B09KRQ18Z1', 'B0B4JNTNX1']


In [29]:
missing_asins

['B00005QI1H',
 'B0017WEH1S',
 'B0018CJ7G2',
 'B00192H1KA',
 'B004AAIQ1G',
 'B0092KKTQ4',
 'B009WUPGQC',
 'B00BX8RRJA',
 'B00OPZR8L0',
 'B079659G3W',
 'B07TLHM5S8',
 'B085QPYTJ8',
 'B08XYQL2VF',
 'B099J4VRS2',
 'B09JP1HXN8',
 'B09JP21WCK',
 'B09KRQ18Z1',
 'B0B4JNTNX1']

In [30]:
retry_image_missing_asins = df_retry_image[df_retry_image["has_retry_image"] == False]["asin"]

In [31]:
retry_image_missing_asins

0     B009WUPGQC
2     B09KRQ18Z1
3     B07TLHM5S8
5     B0018CJ7G2
6     B0092KKTQ4
9     B00005QI1H
14    B0017WEH1S
18    B079659G3W
20    B00OPZR8L0
21    B099J4VRS2
25    B085QPYTJ8
26    B004AAIQ1G
28    B09JP21WCK
29    B00192H1KA
30    B08XYQL2VF
31    B09JP1HXN8
32    B0B4JNTNX1
33    B00BX8RRJA
Name: asin, dtype: str

In [32]:
set(missing_asins) == set(retry_image_missing_asins)

True

## Đống hình này thật sự không có nên sẽ điền image_feat mặc định lúc train
## Lưu lại đánh dấu cái nào có image

In [33]:
df_meta_with_image.columns

Index(['main_category', 'title', 'average_rating', 'rating_number', 'features',
       'description', 'price', 'images', 'videos', 'store', 'categories',
       'details', 'asin', 'bought_together', 'subtitle', 'author',
       'has_image'],
      dtype='str')

In [34]:
df_filtered_image = df_meta_with_image.loc[:, ["title", "asin"]].copy()

In [35]:
df_filtered_image.info()

<class 'pandas.DataFrame'>
RangeIndex: 35997 entries, 0 to 35996
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   title   35997 non-null  str  
 1   asin    35997 non-null  str  
dtypes: str(2)
memory usage: 4.7 MB


In [36]:
df_filtered_image.shape

(35997, 2)

In [37]:
df_filtered_image.head(1)

,title,asin
0,"Chicco Viaro Travel System, Teak",B01C4319LO


In [38]:
# Tạo cột mới thay cho has_image vì sợ trùng tên
# cột mới sẽ là img path nào None là k có hình

# Điền giá trị mặc định cho cột image_path là: step0_clean_data/filtered_images
df_filtered_image["image_path"] = df_filtered_image.apply(lambda row: f"step0_clean_data/filtered_images/{row['asin']}.jpg", axis=1)

# Nếu không có hình
df_filtered_image.loc[df_filtered_image["asin"].isin(set(missing_asins)), "image_path"] = None

In [40]:
df_filtered_image["image_path"]

0        step0_clean_data/filtered_images/B01C4319LO.jpg
1        step0_clean_data/filtered_images/B0083SXABC.jpg
2        step0_clean_data/filtered_images/B07JM4RK9T.jpg
3        step0_clean_data/filtered_images/B08F1VWF5P.jpg
4        step0_clean_data/filtered_images/B01DDDXTA8.jpg
                              ...                       
35992    step0_clean_data/filtered_images/B0CHYPBD2Z.jpg
35993    step0_clean_data/filtered_images/B0BR6CWGKL.jpg
35994    step0_clean_data/filtered_images/B0965ZFFHW.jpg
35995    step0_clean_data/filtered_images/B0C614K38T.jpg
35996    step0_clean_data/filtered_images/B0BZ5NMGLK.jpg
Name: image_path, Length: 35997, dtype: str

In [41]:
df_filtered_image.info()

<class 'pandas.DataFrame'>
RangeIndex: 35997 entries, 0 to 35996
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   title       35997 non-null  str  
 1   asin        35997 non-null  str  
 2   image_path  35979 non-null  str  
dtypes: str(3)
memory usage: 6.6 MB


In [42]:
df_filtered_image.to_parquet(os.path.join(PATH, "step4_image_feature_extract", "df_filtered_image.parquet"), index=False)